# 素体の 3D モデル用: Hunyuan3D-2mv を Colab で動かす

novelai-userscripts-example の素体の 3D モデル(漫画のポーズの参考)で、素体の正面・横・後ろの絵から 3D の形を作るための
リモートの GPU です([Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2) の多視点版 2mv。形だけで、テクスチャは塗りません)。

1. メニューの「ランタイム → ランタイムのタイプを変更」で **L4**(か A100)を選ぶ(1体 2〜3 分、GPU メモリは 6GB ほど)
2. 上から順にセルを実行する。準備のセルの最後でセッションが再起動する(「クラッシュ」と出るが正常)。再起動したら、
   準備のセルはやり直さず、次のセルから続ける
3. 最後のセルに出る 2 行を、アプリの `.env` に書いてバックエンドを再起動する

使い終わったら「ランタイム → ランタイムを接続解除して削除」で止めてください(止めるまでコンピューティング ユニットが減ります)。

Hunyuan3D-2 は Tencent Hunyuan Community License です(EU・英国・韓国は対象外)。背景消し(rembg)は非商用の RMBG-2.0 を使います。
自分で使うだけにしてください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Hunyuan3D-2 を入れる(5〜10 分)

形を作る部分だけを使うので、テクスチャ用の部品はビルドしません。

In [ ]:
%cd /content
![ -d Hunyuan3D-2 ] || git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .
# 新しい transformers(5)では、画像の部品(DINOv2)の重みの名前が変わり、配布の重みを読めない
!pip install -q "transformers>=4.46,<5"
# transformers を戻すと huggingface-hub も古くなり、新しい diffusers と合わなくなる
!pip install -q "diffusers>=0.30,<0.36"
!pip install -q fastapi uvicorn python-multipart
# 「pip's dependency resolver」の ERROR は、Colab に入っている別のライブラリ(cudf・gradio など)の警告で、ここでは使わない。
# pip で numpy などが入れ替わったので、読み込み直すためにセッションを再起動する(「クラッシュ」と出るが正常)。
# 再起動のあとは、このセルをやり直さず、次のセルから実行する
import os
os.kill(os.getpid(), 9)

## 2. 合言葉付きのサーバーを起動する

アプリから絵を受け取って GLB を返すサーバーです。Colab の中だけで待ち、合言葉(トークン)が無い呼び出しは断ります。
起動するとすぐにモデルを読み込み始めます(初回は重みの取得で数分)。

In [ ]:
import json, os, secrets, subprocess, time, urllib.request

BRANCH = "main"
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/hirotoitpost/novelai-userscripts-example/{BRANCH}/scripts/model3d_server.py",
    "/content/Hunyuan3D-2/model3d_server.py",
)
TOKEN = globals().get("TOKEN") or secrets.token_urlsafe(32)
server = subprocess.Popen(
    ["python", "model3d_server.py", "--port", "8191"],
    cwd="/content/Hunyuan3D-2", env={**os.environ, "MODEL3D_TOKEN": TOKEN},
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
# モデルを読み込み終わるまで待つ(最長 15 分)
start = time.time()
while time.time() - start < 15 * 60:
    if server.poll() is not None:
        raise RuntimeError("サーバーが止まりました。下の「ログを見る」のセルで理由を確かめてください。")
    try:
        req = urllib.request.Request("http://127.0.0.1:8191/health", headers={"Authorization": f"Bearer {TOKEN}"})
        health = json.loads(urllib.request.urlopen(req, timeout=2).read())
        if health["loaded"]:
            print(health)
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("15 分たってもモデルを読み込み終わりません。ログを確かめてください。")

## 3. 外から使えるようにする(cloudflared)

URL はこのセルを実行するたびに変わります。

In [ ]:
import re, subprocess, threading, time, urllib.request

urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared"
)
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8191"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
start = time.time()
while url is None and time.time() - start < 60:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if match:
        url = match.group(0)
if url is None:
    tunnel.kill()
    raise RuntimeError("60 秒たっても cloudflared の URL が出ませんでした。このセルをもう一度実行してください。")
# 出力を読み続けないと cloudflared が詰まって止まるので、残りは捨て続ける
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("アプリの .env に書く 2 行:")
print(f"MODEL3D_BASE_URL={url}")
print(f"MODEL3D_TOKEN={TOKEN}")

## 4. ログを見る

うまくいかないときや、使い終わったあとに。終わったら「ランタイム → ランタイムを接続解除して削除」を選んでください。

In [ ]:
!tail -n 30 /content/server.log